In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [8]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="qwen3:4b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)
agent = create_agent(
    model=model,
    tools=[sql_query]
)

In [10]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [11]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='81ed310d-8c53-4141-b323-7b7db34f8902'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 618, 'prompt_tokens': 147, 'total_tokens': 765, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-414', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038af-1329-7521-914f-415e76ebdc67-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT name FROM artists WHERE name LIKE 'S%' ORDER BY popularity DESC LIMIT 1"}, 'id': 'call_xrhadtld', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 147, 'output_tokens': 618, 'total_tokens': 765, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content="Error: (sql

In [12]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT a.Name FROM Artist a JOIN Track t ON a.ArtistId = t.ArtistId WHERE a.Name LIKE 'S%' GROUP BY a.ArtistId ORDER BY COUNT(t.TrackId) DESC LIMIT 1
